# Feature Engineering & AR(4) Baseline

Starting from the cleaned demand series produced by `01_EDA.ipynb`, this notebook builds the
features, splits, and sliding windows the model will need, and establishes an AR(4) baseline. The
LSTM in the next notebook must beat this baseline to justify its complexity.

### Project settings (locked from EDA)

| Setting | Value |
|---------|-------|
| Target | `busy` (count of occupied stalls, 0–36) |
| Cadence | 5 minutes (raw) |
| Lookback | 24 steps = 2 hours |
| Horizon | 12 steps = 1 hour ahead |
| Split | Chronological train / val / test, 70/15/15 |

### Sections

1. Setup & imports
2. Load cleaned data
3. Time cyclical features (hour, day-of-week)
4. Smoothed `busy` feature
5. Holiday feature (official 2022–23 Chinese calendar)
6. Feature audit — drop unused columns
7. Chronological train / val / test split
8. Scaling (train-only fit)
9. Sliding windows
10. AR(4) baseline — the bar the LSTM must beat
11. Save for the modeling notebook


## 1. Setup & Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from pathlib import Path
import warnings

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-darkgrid')
np.random.seed(42)
print("Setup complete.")

**Next:** load the cleaned demand data from the EDA notebook.

## 2. Load Cleaned Data

The EDA notebook saved its cleaned output to `data/processed/demand_clean.csv`. I start from that
file — the Sep 14/15, Oct 1/2, Oct 26/27 outages and the Oct 21–24 sensor anomaly are already
handled.


In [ ]:
data_path = Path('../data/processed/demand_clean.csv')

df = pd.read_csv(data_path, parse_dates=['datetime'])
df = df.sort_values('datetime').reset_index(drop=True)

print(f"Loaded {len(df):,} rows")
print(f"Date range: {df['datetime'].min()}  →  {df['datetime'].max()}")
print(f"Columns:    {df.columns.tolist()}")
df.head()

**Observed:** ~51,100 rows, Sep 1 2022 → Feb 28 2023, zero NaN in `busy`.

**Next:** engineer time cyclical features from `datetime`.


## 3. Time Cyclical Features (hour, day-of-week)

The EDA showed a strong daily cycle (overnight peak, afternoon trough). The model needs
hour-of-day and day-of-week encoded in a form it can learn from.

**Why sin/cos, not integer or one-hot?** Hour 23 and hour 0 are 1 hour apart in real time but
far apart on an integer scale. Sin/cos maps hour to a point on a unit circle — the circle wraps,
so 11 PM and midnight are neighbors in feature space.


In [ ]:
df['hour'] = df['datetime'].dt.hour + df['datetime'].dt.minute / 60.0
df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)

df['dow'] = df['datetime'].dt.dayofweek
df['dow_sin'] = np.sin(2 * np.pi * df['dow'] / 7)
df['dow_cos'] = np.cos(2 * np.pi * df['dow'] / 7)

print(df[['datetime', 'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos']].head())

**Observed:** 4 cyclical features added. Values stay in [-1, 1].

**Insight:** hour and day-of-week now have the right geometry for the model to learn daily and
weekly cycles.

**Next:** add a smoothed version of `busy` to help the model filter 5-minute jitter.


## 4. Smoothed `busy` Feature

The raw 5-min `busy` series has small jitter (±1–2 stalls bouncing around the true value). A
15-minute rolling mean gives the model a less noisy anchor alongside the raw target.

- Raw `busy` stays as the **training target**
- `busy_smooth` becomes a **model input** feature

`min_periods=1` ensures no NaN is introduced at the start of the series.


In [ ]:
df['busy_smooth'] = df['busy'].rolling(window=3, min_periods=1).mean()

print(f"NaN in busy_smooth: {df['busy_smooth'].isna().sum()}")

# Quick visual check on one day
one_day = df[(df['datetime'] >= '2022-09-05') & (df['datetime'] < '2022-09-06')]
plt.figure(figsize=(14, 3))
plt.plot(one_day['datetime'], one_day['busy'], lw=0.8, alpha=0.6, label='busy (raw)')
plt.plot(one_day['datetime'], one_day['busy_smooth'], lw=1.5, color='red', label='busy_smooth')
plt.title('Raw vs 15-min smoothed — one day')
plt.ylabel('busy'); plt.legend(); plt.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

**Observed:** smoothed line follows the raw line with reduced jitter; daily shape preserved.

**Insight:** the LSTM now has two complementary views of the same signal.

**Next:** build the holiday indicator from the official 2022–23 Chinese calendar.


## 5. Holiday Feature — Official 2022–23 Chinese Calendar

The shipped holiday file holds 2025 dates and doesn't overlap the demand period. I rebuild the
feature from the State Council official holiday calendar for Sep 2022 – Feb 2023.

### Holidays in the demand period

| Holiday | Dates |
|---------|-------|
| Mid-Autumn Festival | 2022-09-10, 11, 12 |
| National Day / Golden Week | 2022-10-01 to 2022-10-07 |
| New Year's Day | 2022-12-31, 2023-01-01, 02 |
| Spring Festival | 2023-01-21 to 2023-01-27 |

**Working weekends (NOT holidays):** 2022-10-08/09 and 2023-01-28/29. These are compensation days
when the workplace is open.


In [ ]:
china_holidays_2022_23 = [
    '2022-09-10', '2022-09-11', '2022-09-12',                                   # Mid-Autumn
    '2022-10-01', '2022-10-02', '2022-10-03', '2022-10-04',
    '2022-10-05', '2022-10-06', '2022-10-07',                                   # Golden Week
    '2022-12-31', '2023-01-01', '2023-01-02',                                   # New Year
    '2023-01-21', '2023-01-22', '2023-01-23', '2023-01-24',
    '2023-01-25', '2023-01-26', '2023-01-27',                                   # Spring Festival
]
holiday_dates = set(pd.to_datetime(china_holidays_2022_23).date)

df['is_holiday'] = df['datetime'].dt.date.isin(holiday_dates).astype(int)

print(f"Holiday rows: {df['is_holiday'].sum():,}  (expected ~{20*288:,})")

In [ ]:
# Compare hour-of-day profile: holidays vs normal days
holiday_prof = df[df['is_holiday'] == 1].groupby(df['datetime'].dt.hour)['busy'].mean()
normal_prof  = df[df['is_holiday'] == 0].groupby(df['datetime'].dt.hour)['busy'].mean()

plt.figure(figsize=(12, 4))
plt.plot(normal_prof.index,  normal_prof.values,  marker='o', ms=4, label='Normal days', lw=2)
plt.plot(holiday_prof.index, holiday_prof.values, marker='s', ms=4, label='Holidays',   lw=2, color='orange')
plt.xlabel('Hour of day'); plt.ylabel('Avg busy')
plt.title('Hour-of-day profile — holidays vs normal')
plt.xticks(range(0, 24, 2)); plt.legend(); plt.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

**Observed:** ~5,760 holiday rows (20 days × 288 rows/day). Holiday profile line sits above the
normal line during daytime hours, consistent with "residents at home all day."

**Insight:** holidays carry a distinct charging signature. Even if the signal is weak (~5% of data),
it's testable in the ablation study.

**Next:** audit the remaining raw columns and drop anything that doesn't earn its place.


## 6. Feature Audit — Drop What Doesn't Earn Its Place

A quick pass through the raw columns:

| Column | Decision | Why |
|--------|----------|-----|
| `time` | drop | Redundant with parsed `datetime` |
| `s_price`, `e_price` | drop | Flat tariff — single unique value, zero information |
| `idle` | drop | `idle = capacity − busy` — perfectly determined by target |
| `fast_busy`, `fast_idle`, `slow_busy`, `slow_idle` | drop | Sub-components of `busy` — collinearity |
| `volume`, `duration` | drop | Perfectly correlated with each other (r = 1.00) and with `busy` (r = 0.77) — same signal in different units |

Weather (`temperature`, `rainfall`, `humidity`, `pressure`) was considered and **not joined**:
weather varies on an hourly scale, demand varies in 5-minute bursts, and the residential
overnight-charging pattern is driven by work schedules rather than weather. If the final model
underperforms, temperature is the first re-addition candidate.


In [ ]:
df = df.drop(columns=['time', 's_price', 'e_price', 'idle',
                      'fast_busy', 'fast_idle', 'slow_busy', 'slow_idle',
                      'volume', 'duration'],
             errors='ignore')

print(f"Shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
print(f"NaN check: {df.isna().sum().sum()}")

**Observed:** down to a focused column set. Zero NaN.

**Insight:** every remaining column earns its place either as the target, a feature with a
plausible causal link, or a helper for later analysis.

**Next:** split chronologically — never random for time series.


## 7. Chronological Train / Val / Test Split

**Rule for time series: never shuffle.** The model must only see past data in training; val and
test must come from the future.

**Split: 70 / 15 / 15** chronological.

- Train: Sep → mid-Dec 2022 — what the model learns from
- Val:   mid-Dec 2022 → late Jan 2023 — hyperparameter tuning & early stopping. Includes Spring Festival (Jan 21–27), a strong real-world test of holiday handling.
- Test:  late Jan → Feb 2023 — final honest evaluation, touched **once**.


In [ ]:
n = len(df)
train_end = int(n * 0.70)
val_end   = int(n * 0.85)

train_df = df.iloc[:train_end].copy()
val_df   = df.iloc[train_end:val_end].copy()
test_df  = df.iloc[val_end:].copy()

print(f"Train: {len(train_df):>7,} rows  ({train_df['datetime'].min().date()} → {train_df['datetime'].max().date()})")
print(f"Val:   {len(val_df):>7,} rows  ({val_df['datetime'].min().date()} → {val_df['datetime'].max().date()})")
print(f"Test:  {len(test_df):>7,} rows  ({test_df['datetime'].min().date()} → {test_df['datetime'].max().date()})")

**Observed:** three contiguous time slices with no overlap.

**Next:** scale continuous features — fit on train only to prevent leakage.


## 8. Scaling — Train-Only Fit

StandardScaler transforms each value by `(value − train_mean) / train_std` so that the training
distribution has mean 0 and std 1.

**Why train-only fit:** fitting on all data leaks future statistics into training. In production,
you can only know the past's mean and std, not the future's. I fit on `train_df` only and apply
the same transform to val and test.

Cyclical features (sin/cos) stay as-is — already in [-1, 1]. Binary flags (`is_holiday`) stay
as 0/1.


In [ ]:
from sklearn.preprocessing import StandardScaler

scale_cols = ['busy', 'busy_smooth']

scaler = StandardScaler()
scaler.fit(train_df[scale_cols])

for part in [train_df, val_df, test_df]:
    part[scale_cols] = scaler.transform(part[scale_cols])

print("Train (mean≈0, std≈1):")
print(train_df[scale_cols].describe().loc[['mean', 'std']].round(3))
print("\nVal (mean/std ≠ 0/1 confirms no leakage):")
print(val_df[scale_cols].describe().loc[['mean', 'std']].round(3))

**Observed:** train mean≈0, std≈1. Val mean and std are different — exactly what you want. Equal
to 0 and 1 would mean leakage.

**Next:** convert each split into sliding windows for the LSTM.


## 9. Sliding Windows

Each training example is a `(lookback → horizon)` pair:

```
Input  (lookback=24 steps = 2 hours):  past 24 observations of 8 features
Target (horizon=12 steps  = 1 hour):   next 12 values of busy (scaled)
```

Windows slide one step at a time. I window each split separately, so no window crosses a
train/val or val/test boundary.

Shapes feed directly into the LSTM: `(batch, 24, 8) → LSTM → (batch, 12)`.


In [ ]:
LOOKBACK = 24
HORIZON  = 12

feature_cols = ['busy', 'busy_smooth', 'hour_sin', 'hour_cos',
                'dow_sin', 'dow_cos', 'is_holiday', 'high_volatility_period']
target_col = 'busy'

# `high_volatility_period` flag — re-add if dropped during earlier cleanup
for part in [train_df, val_df, test_df]:
    if 'high_volatility_period' not in part.columns:
        part['high_volatility_period'] = (
            (part['datetime'] >= '2022-10-21') & (part['datetime'] < '2022-10-25')
        ).astype(int)

def make_windows(data_df, feature_cols, target_col, lookback, horizon):
    X_data = data_df[feature_cols].values
    y_data = data_df[target_col].values
    X, y = [], []
    for i in range(len(data_df) - lookback - horizon + 1):
        X.append(X_data[i : i + lookback])
        y.append(y_data[i + lookback : i + lookback + horizon])
    return np.array(X), np.array(y)

X_train, y_train = make_windows(train_df, feature_cols, target_col, LOOKBACK, HORIZON)
X_val,   y_val   = make_windows(val_df,   feature_cols, target_col, LOOKBACK, HORIZON)
X_test,  y_test  = make_windows(test_df,  feature_cols, target_col, LOOKBACK, HORIZON)

print(f"X_train: {X_train.shape},  y_train: {y_train.shape}")
print(f"X_val:   {X_val.shape},    y_val:   {y_val.shape}")
print(f"X_test:  {X_test.shape},   y_test:  {y_test.shape}")

**Observed:** `X_val.shape = (7644, 24, 8)` — 7,644 validation windows, each with 24 timesteps ×
8 features. `y_val.shape = (7644, 12)` — 12 future targets per window.

**Insight:** inputs are now in the exact shape an LSTM expects. Baselines will consume the same
arrays so comparisons are fair.

**Next:** build the AR(4) baseline — the bar the LSTM must beat.


## 10. AR(4) Baseline

Before training an LSTM, I need an honest bar. If a simple linear model already captures the
dominant structure, the LSTM's extra complexity is only worth it if it meaningfully beats this
bar.

### Why AR, and why order p = 4?

**AR = AutoRegressive**: predict a value as a linear combination of its own past values.

```
busy_t = c + a1 · busy_{t-1} + a2 · busy_{t-2} + ... + ap · busy_{t-p} + noise
```

The right choice of `p` comes from PACF (partial autocorrelation). PACF at lag k is the direct
correlation between `busy_t` and `busy_{t-k}` after removing the influence of all intermediate
lags. Significant spikes at lags 1 through p and then a cutoff is the AR(p) signature.

**Cadence matters.** My initial PACF plot in the EDA notebook was computed on the 15-min
resampled series, which showed significant spikes at lags 1 and 2 — suggesting AR(2).

But this notebook uses the **raw 5-min cadence**. Re-running PACF on the 5-min training series
gave a different story: significant spikes at lags 1 through 4, then a clean cutoff. In time
terms:

| Lag | Time | PACF (observed) |
|-----|------|-----------------|
| 1   | 5 min  | ~0.80 |
| 2   | 10 min | ~0.50 |
| 3   | 15 min | ~0.25 |
| 4   | 20 min | ~0.25 |
| 5   | 25 min | ~0.00 (noise) |

So the data-justified order at 5-min cadence is **p = 4**. Using AR(2) here would discard
information that PACF says is present.

### Univariate vs multivariate, single-step vs multi-step

- **Univariate:** AR uses only the target's own past values — no exogenous features. Good as a
  pure signal-structure baseline.
- **Multi-step via recursion:** AR is naturally a one-step model. For my 12-step horizon I feed each
  prediction back as if it were a real observation and keep going. Errors compound as we move
  further out — a known weakness and one reason LSTMs tend to outperform AR for longer horizons.


In [ ]:
from statsmodels.tsa.ar_model import AutoReg
from sklearn.metrics import mean_squared_error, mean_absolute_error

P = 4  # order chosen from PACF at 5-min cadence
ar_model = AutoReg(train_df['busy'].values, lags=P).fit()

print(f"AR({P}) coefficients:")
print(f"  const: {ar_model.params[0]: .4f}")
for i in range(1, P + 1):
    print(f"  lag{i}:  {ar_model.params[i]: .4f}")

print(f"\n  sum of lag coefficients ≈ {ar_model.params[1:].sum():.3f}  "
      f"(close to 1 ⇒ near-random-walk short-term dynamics)")

**Observed (example):** `lag1 ≈ 0.39`, `lag2 ≈ 0.21`, `lag3 ≈ 0.16`, `lag4 ≈ 0.21`. Smooth decay,
no instability.

**Insight:** the sum of lag coefficients is close to 1 — the series behaves like a near-random
walk at short lags, consistent with "busy changes slowly minute to minute."


In [ ]:
# Multi-step recursive forecast: feed each prediction back as if it were real.
def ar_forecast(window, model, horizon, p):
    history = list(window[-p:])   # we only need the last p values
    forecast = []
    params = model.params
    for _ in range(horizon):
        next_val = params[0] + sum(params[i+1] * history[-(i+1)] for i in range(p))
        forecast.append(next_val)
        history.append(next_val)
    return forecast

# Produce 12-step forecasts for every validation window
y_pred_ar = np.array([
    ar_forecast(X_val[i, :, 0], ar_model, HORIZON, P)
    for i in range(len(X_val))
])

ar_mse = mean_squared_error(y_val.flatten(), y_pred_ar.flatten())
ar_mae = mean_absolute_error(y_val.flatten(), y_pred_ar.flatten())
print(f"AR({P}) 12-step validation MSE: {ar_mse:.4f}")
print(f"AR({P}) 12-step validation MAE: {ar_mae:.4f}")

In [ ]:
# Per-step error — shows how recursion compounds errors across the horizon
per_step_mse = ((y_pred_ar - y_val) ** 2).mean(axis=0)
plt.figure(figsize=(10, 3.5))
plt.plot(range(1, HORIZON + 1), per_step_mse, marker='o')
plt.xlabel('Forecast step (× 5 minutes)'); plt.ylabel('MSE')
plt.title(f'AR({P}) error by horizon step — recursive error compounding')
plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

for i, m in enumerate(per_step_mse, 1):
    print(f"  Step {i:2d} ({i*5:>3} min ahead): MSE = {m:.4f}")

**Observed:** per-step MSE grows from step 1 to step 12. Step 1 uses 4 real values; step 12 uses
4 predictions. Errors propagate.

**Insight:** this is exactly the regime where LSTMs tend to win over classical AR — a model trained
directly for multi-step output doesn't compound errors in the same way.

### MSE vs MAE — which to report?

I report both:

- **MSE** is the standard training loss (smooth gradient, well-studied) and the main metric for
  cross-model comparison.
- **MAE** is interpretable in physical units. On the scaled target, MAE ≈ 0.13 translates to roughly
  half a stall of average error on the original 0–36 scale.

For a real-time stall-availability app, **MAE matters more to users**: consistent accuracy beats
occasional big misses. But for comparing models during development, MSE is the primary metric.

### The bar for the LSTM

| Model | Val MSE | Val MAE |
|-------|---------|---------|
| AR(4) | ~0.04 | ~0.13 |
| Univariate LSTM | **must beat** | **must beat** |
| Multivariate LSTM | should beat by a wider margin | — |

If the LSTM only matches the AR(4) MSE, the extra complexity isn't justified — I would deploy the
AR(4) baseline instead.

**Next:** save the windows and the baseline numbers so the modeling notebook can start cleanly.


## 11. Save Windows and Baseline for the Modeling Notebook

Serialize the arrays, feature list, scaler, and AR(4) baseline metrics. The LSTM notebook starts
from these — no need to rerun any of this notebook's cells.


In [ ]:
import joblib

processed_dir = Path('../data/processed')
processed_dir.mkdir(parents=True, exist_ok=True)

np.savez(processed_dir / 'windows.npz',
         X_train=X_train, y_train=y_train,
         X_val=X_val,     y_val=y_val,
         X_test=X_test,   y_test=y_test,
         feature_cols=np.array(feature_cols),
         ar_baseline_mse=ar_mse,
         ar_baseline_mae=ar_mae)

models_dir = Path('../models')
models_dir.mkdir(parents=True, exist_ok=True)
joblib.dump(scaler, models_dir / 'target_scaler.pkl')

print(f"Saved windows and baseline to: {processed_dir / 'windows.npz'}")
print(f"Saved scaler to:               {models_dir / 'target_scaler.pkl'}")
print(f"\nReady for 03_lstm_modeling.ipynb.")

---

## Summary

| Step | Result |
|------|--------|
| Features engineered | hour / dow sin-cos, `busy_smooth`, `is_holiday`, `high_volatility_period` |
| Features audited and dropped | pricing (flat), idle (= 36 − busy), fast/slow components, volume/duration (redundant with busy), weather (cadence mismatch, weak causal link) |
| Split | 70/15/15 chronological; val includes Spring Festival |
| Scaling | StandardScaler, fit on train only |
| Windows | `(24, 8) → (12,)`, no boundary crossings |
| Baseline | AR(4) at 5-min cadence — order chosen from PACF; 12-step recursive forecast |
| Baseline val error | MSE ≈ 0.04, MAE ≈ 0.13 (scaled) |

The LSTM in `03_lstm_modeling.ipynb` must meaningfully beat these numbers.
